# MNIST Addition

We show how DeepLog can be used to implement a well-know NeSy experiment: MNIST Addition.
In this problem we are given a pair of MNIST digits, and the goal is to predict their sum. 
Importantly, there is no direct supervision over the MNIST images, only a pair of images and the sum of the digits is available for training.

We will solve this problem by introducing two core modules: `mnistnet_module` that accepts an image and outputs a probability distribution over the possible digits in the image, and `circuit_module` that from the probability distribution of each digit, computes the probability of their sum.

An interactive version of this notebook is [available here](https://www.kaggle.com/code/robinmanhaeve/deeplog-mnist-addition).

## Network
We define a simple CNN to recognize individual digits.

In [ ]:
import torch.nn as nn


class MNISTNet(nn.Module):
    def __init__(self, n=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8 x 14 x 14
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16 x 7 x 7
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, n),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.classifier(x)
        return x


mnist_net = MNISTNet()

## Dataset
We define the MNIST Addition dataset as a Torch dataset.

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import Subset


class MNISTAddition(Dataset):

    def __init__(self, subset):
        transform = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
        )
        self.dataset = torchvision.datasets.MNIST(
            "data", subset == "train", transform=transform, download=True
        )
        self.subset = subset
        self.n = 2

    def __getitem__(self, index):
        i1 = index * self.n
        im1, l1 = self.dataset[i1]
        im2, l2 = self.dataset[i1 + 1]
        return im1, im2, l1 + l2

    def __len__(self):
        return len(self.dataset) // self.n


train_dataloader = DataLoader(
    Subset(MNISTAddition("train"), range(10000)), batch_size=32, num_workers=0
)
test_dataloader = DataLoader(
    Subset(MNISTAddition("TEST"), range(100)), batch_size=32, num_workers=0
)

In [ ]:
from deeplog import get_network_predicate
from deeplog.formula.deeplogmodulefactory import DeepLogModuleFactory
from deeplog.systems.deepproblog import compile_to_module
from deeplog.systems.deepproblog.engine import SimpleEngine as Engine
from deeplog.systems.deepproblog.program import str_to_rules


code = """
addition(I1,I2,S) :- between(0,9,N1), between(0,9,N2), digit(I1,N1), digit(I2,N2), is(S,+(N1,N2)).
classifier(I,N) :: digit(I,N).
?- addition(i1,i2,S).
"""
program = tuple(str_to_rules(code))

# Create a NetworkPredicate class that wraps the MNIST classifier
ClassifierPredicate = get_network_predicate(
    functor="classifier",
    arity=2,
    structure="probability",
    module=mnist_net,
)


## Shaping the input

The `MNISTAddition` dataset produces a pair of images (and one label, the sum of both), but the `MNISTNet` module currently expects a single digit, not a pair. We could perform two calls to `MNISTNet` to classify both images, but here we will instead demonstrate how to batch the two calls into one single `MNISTNet` evaluation. This requires us to reshape the data, which we can do automatically if we specify the symbolic signature of each module.

The symbolic signature of a {class}`~deeplog.module.deeplog_module.DeepLogModule` comprises the module's input- and output shapes. In case of our dataset, we have a pair of tensors, while the input shape of `MNISTNet` is a single tensor of two images. Using these shapes, we can construct a module that automatically reshapes the data.

In [ ]:
from deeplog import SymTensor
from deeplog import construct_transformation


input0 = ("i1",)
input1 = ("i2",)
data_shape = (SymTensor(input0), SymTensor(input1))

Next, we define the predicate class that describes the symbolic interface of `classifier/2`. [`get_network_predicate`](deeplog.formula.predicates.builtin_predicates.get_network_predicate) wraps `MNISTNet` so it can evaluate digits and produce probabilities for each class.

## Knowledge Circuit

As knowledge we have `digit(1) + digit(2) = digit(3)`. We define

1) `addition(I1,I2,S)` as a predicate that takes two digit images `I1` and `I2` and computes the sum `S`.
2) `digit(I,N)` as a neural predicate with `N` taking values in `[0..9]`.
3) the query `addition(i1,i2,S)` that we want to know the probabilities for.

In [ ]:
factory = DeepLogModuleFactory(
    atom_builders={("classifier", 2, "probability"): ClassifierPredicate}
)

result = Engine().get_query_result(program, factory)
circuit_module = compile_to_module(result, factory)

print("Circuit input shape:", circuit_module.get_input_shape())
print("Circuit output shape:", circuit_module.get_output_shape())

Three relevant observations:

1) The program was compiled by {class}`~deeplog.formula.deeplogmodulefactory.deeplogmodulefactory.DeepLogModuleFactory` into a {class}`~deeplog.circuit.circuit.Circuit`. When we call `to_module`, the {class}`~deeplog.circuit.circuit.Circuit` is exported to a KLay-powered {class}`~deeplog.module.deeplog_module.DeepLogModule`.
2) The output of `circuit_module` corresponds to our query, which was grounded to the possible values for `S`. This means a tensor containing the probability for `addition(i1,i2,0)`, for `addition(i1,i2,1)`, etc.
3) The input of `circuit_module` should be a tensor containing the probabilities for the ground neural facts. In our case, this is a tensor containing the probability of `classifier(i1,0)`, of `classifier(i1,1)`, etc.


### Connection to Expectation

Setting `circuit.deterministic = True` compiles the circuit using the SDD backend, which enables exact weighted model counting in the probability semiring. For boolean formulas, this same compilation is exposed via the **expectation** aggregation operator:

```python
# For a boolean formula, expectation compiles to probability semiring:
expectation_module = factory.create_aggregation(
    "expectation",
    variables,
    [],
    boolean_formula,
)
```

In this MNIST addition example, we're computing conditional probabilities `P(sum=S | digit probs)` for each possible sum, which uses the same deterministic compilation internally. The expectation operator is useful when you want to compute `E[f]` for a single boolean formula `f`.

## Final Architecture

The composed `circuit_module` accepts image tensors and internally runs the classifier to produce digit probabilities, then evaluates the circuit. We reshape the dataset's pair of images with `data_module` and reorder outputs with `prediction_output_module`.

In [ ]:
from deeplog.module import Sequential


data_module = construct_transformation(data_shape, circuit_module.get_input_shape())

prediction_output_shape = SymTensor([f"addition(i1,i2,{s})" for s in range(19)])
prediction_output_module = construct_transformation(
    circuit_module.get_output_shape(), prediction_output_shape
)

complete_module = Sequential(data_module, circuit_module, prediction_output_module)


The output shape of `circuit_module` is not guaranteed to match the order expected from our label. We can define a `prediction_output_shape` and create a {class}`~deeplog.module.deeplog_module.DeepLogModule` that transforms the shape to what we expect, similar to how we created and used the `data_module`.

Next, we use `complete_module` within a PyTorch Lightning Module.

In [ ]:
import logging

import pytorch_lightning as pl
import torch
import torchmetrics

from deeplog.util import fast_dev_run_enabled


pl.utilities.disable_possible_user_warnings()
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)


class LightningDPL(pl.LightningModule):
    def __init__(self, dpl_module: nn.Module, learning_rate: float):
        super().__init__()
        self.module = dpl_module
        self.loss = nn.NLLLoss()
        self.learning_rate = learning_rate
        self.train_accuracy = torchmetrics.classification.Accuracy(
            task="multiclass", num_classes=19
        )
        self.test_accuracy = torchmetrics.classification.Accuracy(
            task="multiclass", num_classes=19
        )
        self.loss_history = []
        self.acc_history = []

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def forward(self, *x) -> torch.Tensor:
        return self.module(*x)

    def training_step(self, batch, batch_idx):
        images1, images2, labels = batch
        logits = self.forward(images1, images2)
        log_prob = torch.log(logits + 1e-9)
        loss = self.loss(log_prob, labels)
        acc = self.train_accuracy(logits, labels)
        self.loss_history.append(loss.detach().cpu().item())
        self.acc_history.append(acc.detach().cpu().item())
        self.log("loss", loss, prog_bar=False, logger=False)
        self.log("train_acc", acc, prog_bar=False, logger=False)
        return loss

    def test_step(self, batch, batch_idx):
        images1, images2, labels = batch
        logits = self.forward(images1, images2)
        self.test_accuracy(logits, labels)
        self.log("test_acc", self.test_accuracy)


pl_model = LightningDPL(complete_module, learning_rate=0.001)

## Training

Now, we can train the model.

In [ ]:
trainer = pl.Trainer(
    fast_dev_run=fast_dev_run_enabled(),
    max_epochs=1,
    enable_progress_bar=False,
    enable_model_summary=False,
    logger=False,
    enable_checkpointing=False,
)
trainer.fit(model=pl_model, train_dataloaders=train_dataloader)

### Training loss

A quick look at the training loss across batches.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def moving_average(values, window):
    values = np.array(values, dtype=float)
    if len(values) < window:
        return values
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / float(window)


window = max(1, len(pl_model.loss_history) // 20)
smoothed_loss = moving_average(pl_model.loss_history, window)
smoothed_acc = moving_average(pl_model.acc_history, window)
x_loss = np.arange(len(smoothed_loss))
x_acc = np.arange(len(smoothed_acc))

fig, ax1 = plt.subplots()
color = "tab:blue"
ax1.set_xlabel("Batch (smoothed)")
ax1.set_ylabel("Training loss", color=color)
ax1.plot(x_loss, smoothed_loss, color=color, label="loss (smoothed)")
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
color = "tab:green"
ax2.set_ylabel("Training accuracy", color=color)
ax2.plot(x_acc, smoothed_acc, color=color, label="train_acc (smoothed)")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()
plt.show()

## Testing

Finally, we evaluate our trained model on the MNIST test dataset.

In [ ]:
trainer.test(model=pl_model, dataloaders=test_dataloader)